# 04 · Hybrid search and Reciprocal Rank Fusion

After this notebook you can run a lexical (BM25) and a dense (embedding) retriever side by side, explain why their scores cannot simply be added, implement Reciprocal Rank Fusion from scratch, measure the gain on labelled questions, and wire the same thing up with LangChain's `EnsembleRetriever`.

**Time:** ~25 min · **Runs:** offline on CPU in ~1 min · **Needs:** [BM25 from scratch](02_bm25_from_scratch.ipynb), [vector search](03_vector_search_faiss_hnsw.ipynb)

## Why this matters in interviews

- Hybrid retrieval (BM25 + dense, fused) is the **production default**; "how do you combine the two lanes?" is the follow-up, and the expected answer is RRF with k ≈ 60 - plus *why* not a weighted sum of scores.
- You should be able to write RRF in five lines on a whiteboard and do a worked example by hand.
- The honest nuance scores points: hybrid is mainly a **recall** tool (more right answers somewhere in the top k); precision at rank 1 is the reranker's job ([notebook 05](05_reranking.ipynb)).

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `rg07` | What is hybrid search and how do you combine the two lanes? |
| `rg45` | What is reciprocal rank fusion and why is it the default? |
| `rg02` | Why is embedding-based semantic search not enough for production RAG? |
| `ve13` | Lexical versus dense retrieval - what is the honest comparison? |
| `rg49` | What is the difference between retrieval score and confidence? |
| `rg15` | How do you optimise retrieval quality? Give me the ordered list. |
| `rg00` | Design a RAG system over a knowledge base folder: what language, what packages, and what does the skeleton code look like? |

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi

np.random.seed(0)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

In [ ]:
# The knowledge base: 15 short markdown docs for a fictional company, Acme Cloud (a SaaS analytics product
# that also sells an 'Edge Gateway' device). This cell is data - it is identical in every 09_rag notebook.
KB = {
"refund_policy.md": """# Refund Policy

## Eligibility
Annual plans can be refunded in full within 30 days of purchase. Monthly plans can be cancelled at any time, but the current month is not refunded. Add-on credits and professional-services hours are non-refundable.

## How to request a refund
Open the billing portal, choose the invoice and click "Request refund". Only workspace owners can request refunds.

## Processing time
Approved refunds go back to the original payment method within 5-10 business days. Bank transfers can take up to 15 business days.
""",
"shipping.md": """# Hardware Shipping

## Delivery times
Edge Gateway devices ship from our warehouses in Rotterdam and Dallas. Standard delivery takes 3-5 business days. Express delivery takes 1-2 business days and costs $25.

## Tracking
A tracking link is emailed when the order leaves the warehouse. The first carrier scan can take 24 hours to appear.

## Lost or damaged shipments
If a shipment has not arrived 10 business days after dispatch, report it within 14 days so we can open a carrier investigation. We send a free replacement when the carrier confirms the loss.
""",
"warranty.md": """# Edge Gateway Warranty

## Coverage
Every Edge Gateway has a 2-year limited hardware warranty from the delivery date. It covers manufacturing defects in the device, the power supply and the antennas.

## Not covered
Water damage, drops, unauthorised repairs and cosmetic scratches are not covered by the warranty.

## RMA process
Request an RMA (return merchandise authorisation) in the support portal with the device serial number. Once the RMA is approved we ship a replacement within 7 days, and you send the faulty unit back with the prepaid label.
""",
"api_rate_limits.md": """# API Rate Limits

## Limits per plan
Free: 60 requests per minute. Pro: 600 requests per minute. Business: 1,200 requests per minute. Enterprise: 6,000 requests per minute. Limits apply per API key.

## Best practices
Retry with exponential backoff and jitter, cache responses you read often, and send writes through the bulk endpoint.
""",
"api_errors.md": """# API Error Reference

## ERR-429-A
The per-minute rate limit was exceeded. Wait for the number of seconds in the Retry-After header, then retry.

## ERR-429-B
The daily request quota is exhausted. The quota resets at 00:00 UTC; upgrade the plan for a higher quota.

## ERR-503-C
The API is in scheduled maintenance. Retry after the maintenance window shown on the status page.

## ERR-401-D
The API key was revoked or has expired. Create a new key in Settings > API keys.
""",
"gateway_models.md": """# Edge Gateway Models

## EG-100
The EG-100 is the entry model: 2 Ethernet ports, Wi-Fi 5 and an 8 W maximum power draw. It supports up to 50 sensors.

## EG-200
The EG-200 adds LTE backup: 4 Ethernet ports, Wi-Fi 6 and a 12 W maximum power draw. It supports up to 200 sensors.

## EG-300
The EG-300 is the rugged outdoor model: an IP67 enclosure, 4 Ethernet ports, Wi-Fi 6 and an 18 W maximum power draw. It supports up to 500 sensors.
""",
"oncall_policy.md": """# On-call Policy

## Rotation
Engineers join the on-call rotation after their third month. Each shift lasts one week and hands over on Monday at 09:00 local time. Every shift has a primary and a secondary.

## Response targets
A SEV1 page must be acknowledged within 5 minutes and a SEV2 page within 30 minutes. If a SEV1 page is not acknowledged within 15 minutes, it escalates to the secondary and then to the engineering manager.

## Compensation
On-call engineers receive a stipend of $300 per week, and a day off after any night with more than two pages.
""",
"pricing.md": """# Pricing Plans

## Plans
Free costs $0 and includes 3 users and 1 GB of storage. Pro costs $49 per user per month. Business costs $99 per user per month. Enterprise pricing is custom and starts at 50 users.

## What each plan includes
Single sign-on (SAML and OIDC) is included from the Business plan upward. Audit logs and the 99.95% uptime SLA are included only in Enterprise. All paid plans include email support.

## Billing
Paying annually saves 20% compared with monthly billing. Prices exclude VAT.
""",
"sso_setup.md": """# Single Sign-On Setup

## Supported identity providers
Acme Cloud supports Okta, Microsoft Entra ID (formerly Azure AD) and Google Workspace through SAML 2.0 or OIDC.

## Configuration steps
Go to Settings > Security > Single sign-on, upload the identity provider's metadata XML, and map the email and groups attributes. Test with one account before you enforce SSO for the whole workspace.

## Troubleshooting
SSO-ERR-12 means the SAML assertion was rejected because the clocks differ by more than 5 minutes; sync the identity provider's clock with NTP. SSO-ERR-31 means the email attribute is missing from the assertion.
""",
"data_retention.md": """# Data Retention

## Event data
Raw event data is kept for 30 days on Free, 13 months on Pro and Business, and 25 months on Enterprise. Aggregated reports are kept for as long as the workspace exists.

## Deleted workspaces
When a workspace is deleted, its data is purged from primary storage after 30 days and from backups after a further 35 days.

## Erasure requests
GDPR erasure requests for one person's data are completed within 30 days of identity verification.
""",
"pto_policy.md": """# Paid Time Off

## Allowance
Full-time employees receive 25 days of paid time off per calendar year, plus public holidays. Part-time employees receive a pro-rata allowance.

## Booking
Book absences longer than three days in Workday at least two weeks ahead. Your manager approves the request.

## Carry-over
Up to 5 unused days can be carried into the next year and must be used by 31 March.
""",
"webhooks.md": """# Webhooks

## Delivery and retries
Webhook events are sent as HTTPS POST requests. If your endpoint does not return a 2xx status within 10 seconds, we retry up to 8 times with exponential backoff over 24 hours.

## Verifying signatures
Every request carries an X-Acme-Signature header: an HMAC-SHA256 of the raw body computed with your webhook secret. Compare it in constant time and reject requests older than 5 minutes.
""",
"sla.md": """# Service Level Agreement

## Uptime commitment
Pro and Business are backed by a 99.9% monthly uptime commitment. Enterprise is backed by 99.95%.

## Service credits
If monthly uptime falls below the commitment, you receive a service credit of 10% of the monthly fee; below 99.0% the credit is 25%. Credits are applied to the next invoice.

## How to claim
Open a support ticket within 30 days of the end of the affected month and include the incident dates.
""",
"account_security.md": """# Account Security

## Passwords
Password reset links expire after 60 minutes. After 5 failed sign-in attempts the account is locked for 15 minutes.

## Two-factor authentication
Workspace owners can require two-factor authentication (2FA) for every member. Authenticator apps and hardware security keys are supported; SMS codes are not.
""",
"expenses.md": """# Travel and Expenses

## Meals and hotels
Meals are reimbursed up to $75 per day while travelling. Book hotels through the travel portal; the nightly cap is $200 in most cities.

## Submitting claims
Submit receipts in Workday within 30 days of the expense. Claims over $25 without a receipt are rejected.
""",
}

In [ ]:
# The eval set: each question lists its EVIDENCE - (doc, phrase) pairs that a retrieved chunk must contain.
# Phrases are unique in the KB, so "is the evidence in the retrieved text?" works for any chunking scheme.
QUESTIONS = [
    {"q": "What does ERR-429-A mean?", "kind": "exact id", "evidence": [("api_errors.md", "per-minute rate limit")]},
    {"q": "Our integration keeps failing with ERR-401-D", "kind": "exact id", "evidence": [("api_errors.md", "revoked")]},
    {"q": "How do I fix SSO-ERR-12?", "kind": "exact id", "evidence": [("sso_setup.md", "clocks differ")]},
    {"q": "How many sensors does the EG-300 support?", "kind": "exact id", "evidence": [("gateway_models.md", "500 sensors")]},
    {"q": "How do I get my money back after buying an annual plan by mistake?", "kind": "paraphrase",
     "evidence": [("refund_policy.md", "refunded in full within 30 days")]},
    {"q": "My parcel never showed up. What should I do?", "kind": "paraphrase", "evidence": [("shipping.md", "report it within 14 days")]},
    {"q": "How many vacation days do I get a year?", "kind": "paraphrase", "evidence": [("pto_policy.md", "25 days of paid time off")]},
    {"q": "Do we get compensated for outages?", "kind": "paraphrase", "evidence": [("sla.md", "service credit")]},
    {"q": "Which gateway can be installed outdoors?", "kind": "paraphrase", "evidence": [("gateway_models.md", "rugged outdoor model")]},
    {"q": "Who gets paged when nobody acknowledges a SEV1 alert?", "kind": "keyword", "evidence": [("oncall_policy.md", "engineering manager")]},
    {"q": "How long is event data kept on the Pro plan?", "kind": "keyword", "evidence": [("data_retention.md", "13 months on Pro")]},
    {"q": "What do we get back if the platform is down more than promised?", "kind": "vague", "evidence": [("sla.md", "service credit")]},
    {"q": "What is the API rate limit on the cheapest plan that includes SSO?", "kind": "multi-hop",
     "evidence": [("pricing.md", "from the Business plan upward"), ("api_rate_limits.md", "Business: 1,200 requests")]},
    {"q": "How long do approved refunds take, and how long is the gateway warranty?", "kind": "multi-part",
     "evidence": [("refund_policy.md", "5-10 business days"), ("warranty.md", "2-year")]},
    {"q": "What is the capital of Australia?", "kind": "out-of-scope", "evidence": []},
]
IN_SCOPE = [q for q in QUESTIONS if q["evidence"]]


def has_evidence(text, doc, q):
    """True if this text (from `doc`) contains any of the question's evidence phrases."""
    return any(doc == d and p.lower() in text.lower() for d, p in q["evidence"])


print(f"{len(KB)} docs, {sum(len(t) for t in KB.values()):,} characters; "
      f"{len(QUESTIONS)} questions ({len(IN_SCOPE)} answerable, {len(QUESTIONS) - len(IN_SCOPE)} out of scope)")

In [ ]:
import re


def split_sections(kb):
    """One chunk per '## ' section. The heading path is prepended so every chunk says where it came from."""
    chunks = []
    for doc, text in kb.items():
        title = text.splitlines()[0].lstrip("# ").strip()
        for i, part in enumerate(re.split(r"^## ", text, flags=re.M)[1:]):
            heading, _, body = part.partition("\n")
            path = f"{title} > {heading.strip()}"
            chunks.append({"id": f"{doc}#{i}", "doc": doc, "section": path, "text": f"{path}\n{body.strip()}"})
    return chunks


def is_relevant(chunk, q):
    return has_evidence(chunk["text"], chunk["doc"], q)


CHUNKS = split_sections(KB)
print(f"{len(CHUNKS)} section chunks, avg {sum(len(c['text']) for c in CHUNKS) / len(CHUNKS):.0f} characters")
print("example:", CHUNKS[1]["id"], "->", CHUNKS[1]["text"][:90].replace("\n", " | "), "...")

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import transformers
transformers.utils.logging.disable_progress_bar()          # keep the cell output clean
transformers.logging.set_verbosity_error()
from sentence_transformers import SentenceTransformer


def load_st(name, cls=SentenceTransformer):
    """Load a small Hugging Face model: cached copy first (no network), download only if missing."""
    try:
        return cls(name, local_files_only=True)
    except Exception:
        try:
            return cls(name)                                 # first run only: ~90 MB download
        except Exception as e:
            raise RuntimeError(f"Could not load {name}: connect to the internet once so it can download.") from e


embedder = load_st("sentence-transformers/all-MiniLM-L6-v2")


def encode(texts):
    """Unit-length MiniLM embeddings (384-d), so a dot product IS the cosine similarity."""
    return embedder.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True)


print("embedder: all-MiniLM-L6-v2 |", embedder.get_embedding_dimension(), "dims | max",
      embedder.max_seq_length, "tokens per text (the rest is silently cut off)")

## 1. Two lanes over the same chunks

- **Lexical lane:** BM25 with the analyzer from notebook 02 (lowercase, identifiers kept whole, stopwords dropped). Returns only chunks that share at least one term with the query.
- **Dense lane:** MiniLM embeddings, cosine similarity.

Each lane returns its top `LANE_DEPTH` chunks - in production each lane fetches its top 50-200 from millions; here 10 of 42.

In [ ]:
STOPWORDS = set("""a an the is are was were be been of to in on for with and or but if then so as at by from this that
it its i you we my our your do does did can how what who when which why there their they me""".split())


def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower()) if t not in STOPWORDS]


LANE_DEPTH = 10
bm25 = BM25Okapi([tokenize(c["text"]) for c in CHUNKS])
chunk_vecs = encode([c["text"] for c in CHUNKS])


def bm25_lane(query, depth=LANE_DEPTH):
    scores = bm25.get_scores(tokenize(query))
    order = [i for i in np.argsort(-scores, kind="stable") if scores[i] > 0]    # no shared term = no match
    return order[:depth], scores


def dense_lane(query, depth=LANE_DEPTH):
    scores = chunk_vecs @ encode([query])[0]
    return list(np.argsort(-scores, kind="stable")[:depth]), scores


q = "What does ERR-429-A mean?"
(b_ids, b_sc), (d_ids, d_sc) = bm25_lane(q), dense_lane(q)
print(q)
print("  BM25 :", [(CHUNKS[i]["id"], round(float(b_sc[i]), 2)) for i in b_ids[:3]])
print("  dense:", [(CHUNKS[i]["id"], round(float(d_sc[i]), 3)) for i in d_ids[:3]])

Same question, two opinions: BM25 is certain about the `ERR-429-A` chunk (the rare identifier), the embedding model thinks the neighbouring `ERR-429-B` chunk is closest. Notice the **numbers**: BM25 says ~4, cosine says ~0.3. Can we just add them?

## 2. The score-scale mismatch

In [ ]:
all_bm25 = np.concatenate([bm25_lane(q["q"])[1] for q in IN_SCOPE])
all_cos = np.concatenate([dense_lane(q["q"])[1] for q in IN_SCOPE])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.hist(all_bm25[all_bm25 > 0], bins=30, color="#e8590c")
ax1.set_title(f"BM25 scores (non-zero): 0 to {all_bm25.max():.0f}, unbounded")
ax1.set_xlabel("BM25 score")
ax1.set_ylabel("(question, chunk) pairs")
ax2.hist(all_cos, bins=30, color="#3b5bdb")
ax2.set_title("Cosine similarities: squeezed into roughly -0.1 to 0.9")
ax2.set_xlabel("cosine")
plt.tight_layout()
plt.show()
print(f"BM25 share of pairs with score 0 (no shared term): {np.mean(all_bm25 == 0):.0%}")

A raw sum would be decided almost entirely by BM25. BM25 scores also **move with the corpus** (IDF changes as documents are added) and with query length (more terms, bigger sums), while cosines drift when you change the embedding model. The classic workaround is to normalise each lane per query, then take a weighted sum:

- **min-max:** $(s - \min) / (\max - \min)$ - maps each lane to [0, 1], but one outlier squashes everything else;
- **z-score:** $(s - \mu) / \sigma$ - centred, but assumes a bell shape that BM25's mostly-zero distribution does not have.

In [ ]:
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-12)


def zscore(s):
    return (s - s.mean()) / (s.std() + 1e-12)


def first_hit(order, q):
    return next((r for r, i in enumerate(order, 1) if is_relevant(CHUNKS[i], q)), None)


def weighted_order(query, alpha, norm):
    """alpha * dense + (1 - alpha) * bm25, each normalised per query over all chunks."""
    _, b = bm25_lane(query)
    _, d = dense_lane(query)
    fused = alpha * norm(d) + (1 - alpha) * norm(b)
    return list(np.argsort(-fused, kind="stable"))


def hit_rates(order_fn, ks=(1, 3, 5)):
    ranks = [first_hit(order_fn(q["q"]), q) for q in IN_SCOPE]
    return {f"hit@{k}": np.mean([r is not None and r <= k for r in ranks]) for k in ks} | {
        "MRR": np.mean([1 / r if r else 0 for r in ranks])}


alphas = np.linspace(0, 1, 11)
sweep = pd.DataFrame([{"alpha": a, "norm": name, **hit_rates(lambda s, a=a, f=f: weighted_order(s, a, f))}
                      for name, f in [("min-max", minmax), ("z-score", zscore)] for a in alphas])
print(sweep[sweep["norm"] == "min-max"].round(3).to_string(index=False))

The weight matters a lot - hit@1 swings from 0.50 at alpha = 0 to 0.79 at the best alpha - and that best alpha was found by tuning on these very 14 questions; on another corpus (or next month's corpus) it would be somewhere else. That fragility is the argument for fusing **ranks** instead of scores.

## 3. Reciprocal Rank Fusion from scratch

$$\text{RRF}(d) = \sum_{\text{lanes } \ell} \frac{w_\ell}{k + \text{rank}_\ell(d)} \qquad k = 60,\ w_\ell = 1$$

**In plain English:** every list gives each document a vote worth $1/(60 + \text{rank})$. A document that is 2nd in both lists beats one that is 1st in one list and absent from the other. Scores never enter - only positions - so no normalisation, no calibration, and a lane with weird scores cannot dominate.

In [ ]:
def rrf(ranked_lists, k=60, weights=None):
    """ranked_lists: one list of ids per lane, best first. Returns (ids best first, scores)."""
    weights = weights or [1.0] * len(ranked_lists)
    scores = {}
    for lst, w in zip(ranked_lists, weights):
        for rank, doc in enumerate(lst, start=1):
            scores[doc] = scores.get(doc, 0.0) + w / (k + rank)
    fused = sorted(scores, key=lambda d: -scores[d])            # stable: ties keep first-seen order
    return fused, scores


# Worked example you can do by hand in an interview
lexical = ["refunds", "pricing", "warranty"]
semantic = ["warranty", "refunds", "shipping"]
order, s = rrf([lexical, semantic])
print(pd.DataFrame({"doc": order,
                    "lexical rank": [lexical.index(d) + 1 if d in lexical else "-" for d in order],
                    "semantic rank": [semantic.index(d) + 1 if d in semantic else "-" for d in order],
                    "RRF score": [f"{s[d]:.5f}" for d in order]}).to_string(index=False))
assert np.isclose(s["refunds"], 1 / 61 + 1 / 62) and np.isclose(s["pricing"], 1 / 62)
assert order[:2] == ["refunds", "warranty"], "in both lists beats top of one list"

"refunds" (1st + 2nd) edges out "warranty" (3rd + 1st): $1/61 + 1/62 > 1/63 + 1/61$. "pricing" and "shipping", each in one list only, trail. The constant $k = 60$ flattens the curve - rank 1 is worth only ~3% more than rank 2 - so one lane's confident mistake cannot outvote agreement.

## 4. BM25 vs dense vs hybrid on the eval set

In [ ]:
def hybrid_order(query):
    return rrf([bm25_lane(query)[0], dense_lane(query)[0]])[0]


methods = {"BM25": lambda s: bm25_lane(s)[0], "dense": lambda s: dense_lane(s)[0], "hybrid (RRF)": hybrid_order}
summary = pd.DataFrame({name: hit_rates(fn) for name, fn in methods.items()}).T
print(summary.round(3))
best_weighted = sweep.groupby("norm")[["hit@1", "hit@3"]].max()
print("\nbest weighted-sum settings (tuned on this very eval set - an optimistic number):\n", best_weighted.round(3))

assert summary.loc["hybrid (RRF)", "hit@3"] >= summary[["hit@3"]].drop("hybrid (RRF)").max().item()
assert summary.loc["hybrid (RRF)", "hit@5"] > summary.loc["BM25", "hit@5"]

In [ ]:
ax = summary[["hit@1", "hit@3", "hit@5"]].T.plot(kind="bar", color=["#e8590c", "#3b5bdb", "#2b8a3e"], figsize=(9, 3.8), rot=0)
ax.set_ylim(0, 1.05)
ax.set_ylabel(f"fraction of {len(IN_SCOPE)} questions")
ax.set_title("Hybrid (RRF): the best recall at 3, but not the best rank 1")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for q in IN_SCOPE:
    r = {name: first_hit(fn(q["q"]), q) for name, fn in methods.items()}
    rows.append({"kind": q["kind"], "question": q["q"][:58], **{k: (v or "-") for k, v in r.items()}})
per_query = pd.DataFrame(rows)
print(per_query.to_string(index=False))
pq = per_query.set_index("question")
assert pq.loc["What does ERR-429-A mean?", "hybrid (RRF)"] == 1 and pq.loc["What does ERR-429-A mean?", "dense"] > 1
assert pq.loc["My parcel never showed up. What should I do?", "hybrid (RRF)"] != "-"

A weighted sum *tuned on these same 14 questions* can match RRF's hit@3 and beat its hit@1 - but that is fitting the test set. RRF got its hit@3 with zero tuning, which is what you want on a corpus and a query mix that keep changing.

How to read the per-query table (rank of the first chunk holding the evidence; `-` = not in the lane's list):

- **Exact identifiers** (`ERR-429-A`): dense ranks the evidence 4th, BM25 1st, hybrid 1st - the lexical lane rescues it.
- **Paraphrases** ("parcel never showed up"): BM25 finds nothing relevant, dense finds it, hybrid keeps it in the top few.
- **Hybrid rarely wins rank 1 outright** on paraphrases - a confident-but-wrong BM25 match can push the dense #1 down a place or two. That is why hit@1 does not improve while hit@3 and hit@5 do: fusion buys **recall**; the reranker in the next notebook buys the top position back.
- **The vague question** ("what do we get back if the platform is down...") is missed by every lane - no fusion fixes a query that matches nothing; that needs query rewriting ([notebook 07](07_advanced_rag_query_rewriting_hyde_multiquery.ipynb)).

### Two implementation gotchas

Fusion is only as good as the lists you feed it. Two common mistakes, measured:

In [ ]:
def bad_bm25_lane(query):          # keeps chunks with score 0 -> arbitrary "ranks" for non-matches
    scores = bm25.get_scores(tokenize(query))
    return list(np.argsort(-scores, kind="stable"))


def bad_dense_lane(query):         # no depth limit -> every chunk gets a dense vote
    return list(np.argsort(-(chunk_vecs @ encode([query])[0]), kind="stable"))


variants = {
    "RRF, clean lanes (depth 10, BM25 matches only)": hybrid_order,
    "RRF, BM25 lane keeps zero-score chunks": lambda s: rrf([bad_bm25_lane(s)[:LANE_DEPTH], dense_lane(s)[0]])[0],
    "RRF, both lanes rank the whole corpus": lambda s: rrf([bad_bm25_lane(s), bad_dense_lane(s)])[0],
}
print(pd.DataFrame({k: hit_rates(v) for k, v in variants.items()}).T.round(3))

When BM25 matches nothing ("Do we get compensated for outages?" shares no term with any chunk), its "ranking" is just index order - and RRF happily counts those fake ranks as votes. Always drop non-matches from the lexical lane and fuse **each lane's top-k**, not the whole corpus.

**Score vs confidence (`rg49`):** an RRF score is a rank-based vote, not a probability. For a "do we have anything?" decision use lane-level signals: whether BM25 matched at all, the top cosine, the gap between rank 1 and rank 5, or a reranker score - calibrated on labelled data ([notebook 06](06_rag_pipeline_from_scratch.ipynb) builds that gate).

## 5. The same thing in LangChain: `EnsembleRetriever`

`rg00`'s skeleton uses `EnsembleRetriever(retrievers=[dense, sparse], weights=[0.5, 0.5])`. In LangChain 1.x the legacy retrievers moved to the `langchain-classic` package - the old import path is gone:

In [ ]:
try:
    from langchain.retrievers import EnsembleRetriever        # pre-1.0 path
except ImportError as e:
    print("old import fails in LangChain 1.x:", type(e).__name__, "-", e)

from langchain_classic.retrievers import EnsembleRetriever  # 1.x path
from langchain_community.retrievers import BM25Retriever     # needs rank_bm25
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore


class MiniLMEmbeddings(Embeddings):
    """Wrap our local sentence-transformers model in LangChain's Embeddings interface."""

    def embed_documents(self, texts):
        return encode(texts).tolist()

    def embed_query(self, text):
        return encode([text])[0].tolist()


docs = [Document(page_content=c["text"], metadata={"id": c["id"], "doc": c["doc"]}) for c in CHUNKS]
store = InMemoryVectorStore(embedding=MiniLMEmbeddings())
store.add_documents(docs)
dense_retriever = store.as_retriever(search_kwargs={"k": LANE_DEPTH})
sparse_retriever = BM25Retriever.from_documents(docs, preprocess_func=tokenize)
sparse_retriever.k = LANE_DEPTH
ensemble = EnsembleRetriever(retrievers=[dense_retriever, sparse_retriever], weights=[0.5, 0.5])   # c = 60 by default

for d in ensemble.invoke("What does ERR-429-A mean?")[:3]:
    print(d.metadata["id"], "|", d.page_content[:70].replace("\n", " | "))

`EnsembleRetriever` *is* weighted RRF: each document scores $\sum w_\ell / (c + \text{rank}_\ell)$ with `c = 60`, deduplicated by `page_content` (or by `id_key`). Prove it by fusing the two retrievers' own lists with our function:

In [ ]:
for q in QUESTIONS:
    dense_docs, sparse_docs = dense_retriever.invoke(q["q"]), sparse_retriever.invoke(q["q"])
    ours, _ = rrf([[d.page_content for d in dense_docs], [d.page_content for d in sparse_docs]], k=60, weights=[0.5, 0.5])
    theirs = [d.page_content for d in ensemble.invoke(q["q"])]
    assert ours == theirs, q["q"]
print(f"our rrf() == EnsembleRetriever on all {len(QUESTIONS)} questions")

Two differences from our clean lanes: `BM25Retriever` always returns `k` documents, **including zero-score non-matches** (gotcha 1 above), and the vector store returns its top `k` by cosine. In production you would also swap `InMemoryVectorStore` for Chroma, Qdrant or pgvector, and `MiniLMEmbeddings` for a hosted model - e.g. `llm_setup.langchain_embeddings()` gives an `OpenAIEmbeddings` wired to whichever provider the notebooks use. Nothing downstream changes: that interchangeability is the reason to use the framework.

Elasticsearch and OpenSearch can do the whole thing server-side (an `rrf` retriever / a hybrid query), which avoids shipping two result lists to your application.

## Try it yourself

**Exercise 1.** How sensitive is RRF to its constant? Evaluate `k` in {1, 10, 60, 1000} on hit@1 / hit@3 / MRR.

In [ ]:
# Solution — try it yourself first, then run this
res = {f"k={k}": hit_rates(lambda s, k=k: rrf([bm25_lane(s)[0], dense_lane(s)[0]], k=k)[0]) for k in (1, 10, 60, 1000)}
print(pd.DataFrame(res).T.round(3))
print("small k = trust each lane's top rank more; large k = pure 'how many lists contain it' voting")

**Exercise 2.** Weighted RRF: give the dense lane weight `w` and BM25 `1 - w`, for `w` in {0.3, 0.5, 0.7}. Does favouring either lane help on this eval set?

In [ ]:
# Solution — try it yourself first, then run this
res = {f"dense w={w}": hit_rates(lambda s, w=w: rrf([dense_lane(s)[0], bm25_lane(s)[0]], weights=[w, 1 - w])[0])
       for w in (0.3, 0.5, 0.7)}
print(pd.DataFrame(res).T.round(3))

**Exercise 3.** RRF fuses any number of lists. Add a third lane - BM25 over *stemmed* tokens (`simple_stem` from notebook 02) - and fuse all three.

In [ ]:
# Solution — try it yourself first, then run this
def simple_stem(w):
    for suffix, repl in (("ies", "y"), ("ing", ""), ("ed", ""), ("es", ""), ("s", "")):
        if len(w) > len(suffix) + 3 and w.endswith(suffix):
            return w[: -len(suffix)] + repl
    return w


stem_bm25 = BM25Okapi([[simple_stem(t) for t in tokenize(c["text"])] for c in CHUNKS])


def stem_lane(query):
    s = stem_bm25.get_scores([simple_stem(t) for t in tokenize(query)])
    return [i for i in np.argsort(-s, kind="stable") if s[i] > 0][:LANE_DEPTH]


print(pd.DataFrame({"2 lanes": hit_rates(hybrid_order),
                    "3 lanes (+ stemmed BM25)": hit_rates(lambda s: rrf([bm25_lane(s)[0], dense_lane(s)[0], stem_lane(s)])[0])}).T.round(3))

## Say it in an interview

- **30 seconds:** "I run a lexical lane, BM25 over an inverted index, and a dense lane, embeddings with ANN, each returning its top 50-100, and fuse them with Reciprocal Rank Fusion: each document scores the sum of 1/(60 + rank) over the lists. RRF uses ranks only, so BM25's unbounded scores and cosine's narrow band never have to be calibrated against each other, and it stays stable when the corpus or the embedding model changes. Lexical rescues identifiers and rare terms; dense rescues paraphrases."
- **What it buys:** mainly recall@k. Put a cross-encoder reranker on the fused top 20-50 for precision at rank 1.
- **Numbers:** k = 60 (from the original RRF paper); lane depth 50-200 in production; `EnsembleRetriever` default `c = 60`, weights equal.
- **Traps:** adding raw BM25 and cosine; tuning a weighted sum on the eval set and calling it robust; fusing zero-score BM25 "matches"; fusing whole-corpus rankings; treating an RRF score as a confidence.
- **Follow-ups:** "when would weighted fusion beat RRF?" (when scores are calibrated, e.g. two cross-encoders, or you have enough labelled data to learn the weights); "more than two lanes?" (RRF takes any number: multi-query rewrites, several embedding models, SPLADE).

## Next

- [Reranking](05_reranking.ipynb) - turn the fused recall into precision at rank 1.
- [RAG pipeline from scratch](06_rag_pipeline_from_scratch.ipynb) - hybrid retrieval inside a full grounded-answer pipeline.
- [Advanced RAG: query rewriting, HyDE, multi-query](07_advanced_rag_query_rewriting_hyde_multiquery.ipynb) · [LangChain RAG chain end to end](../13_langchain/05_rag_chain_end_to_end.ipynb) · [RAG skeleton coding round](../17_interview_coding_rounds/01_rag_skeleton.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)